# Machine Learning Notes
## Day 26: Encoding Categorical Data — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Ordinal Encoding & Label Encoding  
> **Difficulty:** Beginner to Intermediate  

---
### Note:
This notebook contains FULLY WORKED SOLUTIONS for every exercise in
**Day26_Encoding_Categorical_Data_Practice_Questions.ipynb**. Use this to check your work
or to study the reference implementation.

### Topics Covered:
1. Identifying nominal vs ordinal features
2. Manual ordinal encoding by hand
3. OrdinalEncoder with explicit category order
4. LabelEncoder on a target column
5. OrdinalEncoder vs LabelEncoder — spotting the right use case
6. Handling unseen categories
7. Mini end-to-end encoding pipeline

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

print('All libraries imported successfully!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Identifying Nominal vs Ordinal Features

In [ ]:
# ============================================================
# ANSWER 1: Classify columns as Nominal or Ordinal
# ============================================================

df = pd.DataFrame({
    'Customer_City':   ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Chennai'],
    'Satisfaction':    ['Low', 'High', 'Medium', 'High', 'Low'],
    'Shirt_Size':      ['S', 'L', 'M', 'XL', 'S'],
    'Payment_Method':  ['Card', 'UPI', 'Cash', 'Card', 'UPI'],
    'Membership_Tier':  ['Silver', 'Gold', 'Platinum', 'Silver', 'Gold']
})

classification = {
    'Customer_City':   'Nominal',   # cities have no inherent rank
    'Satisfaction':    'Ordinal',   # Low < Medium < High
    'Shirt_Size':      'Ordinal',   # S < M < L < XL
    'Payment_Method':  'Nominal',   # Card/UPI/Cash — no order
    'Membership_Tier': 'Ordinal'    # Silver < Gold < Platinum
}

print('Feature Type Classification:')
for col, ftype in classification.items():
    print(f'  {col:18s} -> {ftype}')

print('\nCorrect order for each Ordinal column:')
print('  Satisfaction:    Low < Medium < High')
print('  Shirt_Size:      S < M < L < XL')
print('  Membership_Tier: Silver < Gold < Platinum')

print('\nNominal columns (Customer_City, Payment_Method) should use')
print('One-Hot Encoding instead, since there is no meaningful order.')

---
## Section 2: Manual Ordinal Encoding by Hand

In [ ]:
# ============================================================
# ANSWER 2: Manual Ordinal Encoding
# ============================================================

sizes = pd.Series(['S', 'L', 'M', 'XL', 'S', 'M', 'L'], name='Shirt_Size')

# Step 1: define correct order
size_order = {'S': 0, 'M': 1, 'L': 2, 'XL': 3}

# Step 2 & 3: apply the mapping
encoded_sizes = sizes.map(size_order)

# Step 4: compare
result = pd.DataFrame({
    'Original_Size': sizes,
    'Encoded_Size':  encoded_sizes
})
print(result.to_string(index=False))

print('\nVerification: S=0 (smallest), XL=3 (largest) — order preserved correctly.')

---
## Section 3: OrdinalEncoder with Explicit Category Order

In [ ]:
# ============================================================
# ANSWER 3: OrdinalEncoder — Default vs Explicit Order
# ============================================================

df_tier = pd.DataFrame({
    'Membership_Tier': ['Silver', 'Gold', 'Platinum', 'Silver', 'Gold', 'Platinum']
})

# 1. WITHOUT specifying categories — defaults to alphabetical order
encoder_default = OrdinalEncoder()
encoded_default = encoder_default.fit_transform(df_tier)
print('Default (alphabetical) encoding:')
print('Learned order:', encoder_default.categories_)
# Alphabetically: Gold < Platinum < Silver  -- WRONG! Silver should be lowest.

# 2. WITH explicit correct order: Silver < Gold < Platinum
tier_order = [['Silver', 'Gold', 'Platinum']]
encoder_correct = OrdinalEncoder(categories=tier_order)
encoded_correct = encoder_correct.fit_transform(df_tier)

# 3. Compare side by side
comparison = pd.DataFrame({
    'Original':           df_tier['Membership_Tier'],
    'Default_Encoding':   encoded_default.flatten().astype(int),
    'Correct_Encoding':   encoded_correct.flatten().astype(int)
})
print('\nComparison:')
print(comparison.to_string(index=False))

# 4. Learned categories
print('\nDefault encoder.categories_:', encoder_default.categories_)
print('Correct encoder.categories_:', encoder_correct.categories_)

print('\nConclusion: The default alphabetical order gave Silver=2 (highest)')
print('which is WRONG. The explicit order correctly gives Silver=0 (lowest),')
print('Gold=1, Platinum=2 — matching the true real-world ranking.')

---
## Section 4: LabelEncoder on a Target Column

In [ ]:
# ============================================================
# ANSWER 4: LabelEncoder on Target Variable
# ============================================================

y = pd.Series(['Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes'], name='Will_Purchase')

# 1. Fit and transform
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Compare original vs encoded
comparison = pd.DataFrame({
    'Original': y,
    'Encoded':  y_encoded
})
print('Original vs Encoded:')
print(comparison.to_string(index=False))

# 3. View the learned class mapping
print('\nLearned classes (alphabetical order):', le.classes_)
print('Mapping: No -> 0, Yes -> 1')

# 4. Inverse transform to verify
recovered_labels = le.inverse_transform(y_encoded)
print('\nRecovered labels after inverse_transform:')
print(recovered_labels.tolist())
print('\nMatches original?', list(recovered_labels) == list(y))

---
## Section 5: OrdinalEncoder vs LabelEncoder — Spotting the Right Use Case

In [ ]:
# ============================================================
# ANSWER 5: Choosing the Right Encoder
# ============================================================

# Scenario A: 'Education' is an INPUT feature with a custom order
#   -> Use OrdinalEncoder (input feature + custom order needed)
education_df = pd.DataFrame({
    'Education': ['Graduate', 'School', 'Postgraduate', 'Graduate']
})
edu_order = [['School', 'Graduate', 'Postgraduate']]
edu_encoder = OrdinalEncoder(categories=edu_order)
education_encoded = edu_encoder.fit_transform(education_df)
print('Scenario A — Education (OrdinalEncoder):')
print(education_encoded.flatten())

# Scenario B: 'Disease_Result' is the TARGET variable
#   -> Use LabelEncoder (target variable, single column)
disease_y = pd.Series(['Positive', 'Negative', 'Positive', 'Negative'])
disease_le = LabelEncoder()
disease_encoded = disease_le.fit_transform(disease_y)
print('\nScenario B — Disease_Result (LabelEncoder):')
print('Classes:', disease_le.classes_)
print('Encoded:', disease_encoded)

# Scenario C: TWO input feature columns at once
#   -> Use a SINGLE OrdinalEncoder call with categories for BOTH columns
exp_risk_df = pd.DataFrame({
    'Experience_Level': ['Junior', 'Senior', 'Mid', 'Junior', 'Senior'],
    'Risk_Level':        ['Low', 'High', 'Medium', 'Medium', 'High']
})
multi_order = [
    ['Junior', 'Mid', 'Senior'],    # order for Experience_Level
    ['Low', 'Medium', 'High']       # order for Risk_Level
]
multi_encoder = OrdinalEncoder(categories=multi_order)
exp_risk_encoded = multi_encoder.fit_transform(exp_risk_df)
result_df = pd.DataFrame(exp_risk_encoded,
                          columns=['Experience_Level_enc', 'Risk_Level_enc'])
print('\nScenario C — Experience_Level + Risk_Level (single OrdinalEncoder):')
print(pd.concat([exp_risk_df, result_df], axis=1).to_string(index=False))

print('\nKey takeaway: OrdinalEncoder.categories_ accepts a LIST of lists —')
print('one inner list per column — so multiple ordinal columns can be')
print('encoded together in a single fit_transform() call.')

---
## Section 6: Handling Unseen Categories

In [ ]:
# ============================================================
# ANSWER 6: Handling Unseen Categories Safely
# ============================================================

train_categories = pd.DataFrame({'Tier': ['Silver', 'Gold', 'Platinum', 'Silver']})
test_categories  = pd.DataFrame({'Tier': ['Gold', 'Diamond']})  # 'Diamond' unseen

tier_order = [['Silver', 'Gold', 'Platinum']]

# 1. Fit encoder WITHOUT handle_unknown protection
encoder_strict = OrdinalEncoder(categories=tier_order)
encoder_strict.fit(train_categories)

# 2. Try transforming test data with an unseen category
print('Attempting transform WITHOUT handle_unknown protection:')
try:
    encoder_strict.transform(test_categories)
except ValueError as e:
    print(f'  ERROR raised as expected: {e}')

# 3. Create a new encoder that safely handles unseen categories
encoder_safe = OrdinalEncoder(
    categories=tier_order,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
encoder_safe.fit(train_categories)

# 4. Transform safely
safe_result = encoder_safe.transform(test_categories)
print('\nSafe transform WITH handle_unknown="use_encoded_value":')
comparison = pd.DataFrame({
    'Original':  test_categories['Tier'],
    'Encoded':   safe_result.flatten().astype(int)
})
print(comparison.to_string(index=False))

print('\n\"Diamond\" (unseen during fit) was safely mapped to -1 instead')
print('of raising an error, while \"Gold\" was correctly encoded as 1.')

---
## Section 7: Mini End-to-End Encoding Pipeline

In [ ]:
# ============================================================
# ANSWER 7: Full Encoding Pipeline
# ============================================================

raw = pd.DataFrame({
    'Education':   ['Graduate', 'School', 'Postgraduate', 'Graduate',
                     'Postgraduate', 'School', 'Graduate', 'Postgraduate'],
    'Income_Band': ['Medium', 'Low', 'High', 'Medium',
                     'High', 'Low', 'Medium', 'High'],
    'Approved':    ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes']
})

# Step 1: Split into X and y
X = raw[['Education', 'Income_Band']]
y = raw['Approved']

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# Step 3: Encode X with OrdinalEncoder — fit on training data only
feature_order = [
    ['School', 'Graduate', 'Postgraduate'],   # Education order
    ['Low', 'Medium', 'High']                  # Income_Band order
]
x_encoder = OrdinalEncoder(categories=feature_order)
x_encoder.fit(X_train)

X_train_enc = x_encoder.transform(X_train)
X_test_enc  = x_encoder.transform(X_test)

# Step 4: Encode y with LabelEncoder — fit on training data only
y_encoder = LabelEncoder()
y_encoder.fit(y_train)

y_train_enc = y_encoder.transform(y_train)
y_test_enc  = y_encoder.transform(y_test)

# Step 5: Print final results
print('X_train (encoded):')
print(pd.DataFrame(X_train_enc, columns=['Education_enc', 'Income_Band_enc']))

print('\nX_test (encoded):')
print(pd.DataFrame(X_test_enc, columns=['Education_enc', 'Income_Band_enc']))

print('\ny_train (encoded):', y_train_enc)
print('y_test (encoded): ', y_test_enc)

print('\nLabel classes learned:', y_encoder.classes_)
print('Feature categories learned:', x_encoder.categories_)

print('\nPipeline complete: both X and y are fully numeric and ready')
print('for any scikit-learn model. Note that test data was transformed')
print('using statistics learned ONLY from training data — no leakage.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Concept | What You Learned |
|---|---|
| Nominal data | No natural order — avoid Ordinal/Label Encoding, use One-Hot instead |
| Ordinal data | Has natural order — Ordinal Encoding fits well |
| OrdinalEncoder | For INPUT features, supports multiple columns, `categories=` for custom order |
| LabelEncoder | For the TARGET (y), single column, alphabetical order by default |
| `categories_` / `classes_` | Attributes that reveal the learned order/mapping |
| `inverse_transform()` | Converts encoded values back to original labels |
| Fit rule | Always fit on training data only, transform on test data |
| Unseen categories | Use `handle_unknown='use_encoded_value'` + `unknown_value=-1` |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 26 — Encoding Categorical Data